# ATE Breakdown by Wordplay Type

**Primary author:** Victoria

**Builds on:**
- *05_model_evaluation.ipynb* (Victoria — ATE computation methodology and embedding-loading conventions)
- *wordplay_metadata.ipynb* (Victoria — algorithmic wordplay type detection across the full clue dataset)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

Exploratory analysis (not part of the numbered pipeline) investigating whether
g_stock and g1 show different misdirection patterns — measured as ATE — on
validation clues with different algorithmically verifiable wordplay types.

The notebook has three parts: (1) a descriptive landscape of wordplay type
frequencies within the validation set; (2) a structural comparison between
standard clues and double-definition clues, which have a different
misdirection mechanism; and (3) ATE breakdowns by letterplay type *within
standard clues*, first by individual type and then by grouped category. All
letterplay categories are intersected with `is_standard` so they describe
standard clues only, and every letterplay figure uses `no_letterplay`
(standard clues with no detected letterplay) as its baseline.

Reads validation clues, `wordplay_metadata.csv`, full-vocab wndef embeddings
(Decision 23), and f_clue_val embeddings for both g_stock and g1. Writes
five figures and a results markdown file under `../../outputs/`:
`wp_cooccurrence_heatmap.png`, `wp_ate_structural.png`,
`wp_ate_individual_letterplay.png`, `wp_ate_grouped_letterplay.png`, and
`wp_double_def_comparison.png`.

## §0 — Setup

Standard imports and environment auto-detection. This notebook lives under
`custom_embedding_model/planning/exploration/`, so the component root is two
directories up and the shared `ccc-project/data/` directory is three directories
up. All paths resolve via `pathlib`. `RANDOM_STATE` is pinned so the bootstrap
CI samples in §4 are reproducible.

In [ ]:
# === Imports and configuration
import time
from datetime import date
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    # planning/exploration/ -> custom_embedding_model/ -> ccc-project/
    PROJECT_ROOT = Path("../../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
SHARED_DATA    = PROJECT_ROOT / "data"
WN_DIR         = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
EMBED_DIR      = COMPONENT_ROOT / "data" / "embeddings"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"
FIGURE_DIR     = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:  {env_label}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"WN_DIR:       {WN_DIR}")
print(f"EMBED_DIR:    {EMBED_DIR}")
print(f"OUTPUT_DIR:   {OUTPUT_DIR}")

# §4 uses a bootstrap; fixing the seed keeps CIs identical across re-runs.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print()
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")

## §1 — Load data

Four groups of inputs: (a) validation-split clue rows, (b) algorithmic
wordplay detections per `clue_id`, (c) full-vocab wndef embeddings for both
models (Decision 23 — every validation answer and definition resolves), and
(d) f_clue_val embeddings for both models with their shared row index.

`wordplay_metadata.csv` is keyed on `clue_id` (unique), whereas `clues_val.csv`
has multiple rows per `clue_id` after multi-definition expansion. A left join
from clues_val preserves the row grain, broadcasting the same wordplay flags
across all expansions of a clue.

In [ ]:
# === Load validation clues, wordplay metadata, and join
t0 = time.time()

clues_val = pd.read_csv(
    WN_DIR / "clues_val.csv",
    keep_default_na=False, na_values=[""],
)
print(f"clues_val:          {len(clues_val):,} rows, columns: {list(clues_val.columns)}")

wordplay = pd.read_csv(
    SHARED_DATA / "wordplay_metadata.csv",
    keep_default_na=False, na_values=[""],
)
print(f"wordplay_metadata:  {len(wordplay):,} rows, columns: {list(wordplay.columns)}")

# The 11 boolean wordplay type columns used throughout this notebook.
WP_COLS = [
    "anagram_single_word", "anagram_consec_words",
    "hidden_fwd", "hidden_rev",
    "selection_alt", "selection_alt_rev",
    "selection_firsts", "selection_firsts_rev",
    "selection_lasts", "selection_lasts_rev",
    "double_def",
]
assert set(WP_COLS).issubset(wordplay.columns), "wordplay_metadata missing expected columns"

# Left join clues_val with wordplay flags on clue_id. Every validation clue_id
# must appear in wordplay_metadata (which spans the full source dataset), so
# there should be no nulls in any wordplay column after the join.
joined = clues_val.merge(wordplay[["clue_id"] + WP_COLS], on="clue_id", how="left")
assert joined[WP_COLS].isna().sum().sum() == 0, (
    "Some clue_ids in clues_val did not match a wordplay_metadata row"
)
assert len(joined) == len(clues_val), "Row count changed after join"
print(f"\nJoined frame:       {len(joined):,} rows "
      f"(same grain as clues_val — clue_id is non-unique)")
print(f"Load + join time:   {time.time() - t0:.1f}s")

In [ ]:
# === Load vocabulary, embeddings, and indexes
t0 = time.time()

# Full-vocab wndef vocabulary — canonical row ordering for f_common_wndef.npy.
vocab_wndef = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef.csv",
    keep_default_na=False, na_values=[""],
)
wndef_word_to_row = dict(zip(vocab_wndef["word"], vocab_wndef["row"]))
print(f"vocabulary_wndef: {len(vocab_wndef):,} words")

MODEL_NAMES = ["g_stock", "g1"]

# f_clue_val and f_common_wndef for both models. All arrays are (N, 1024).
# f_common_wndef uses the full-vocab file (53,930 rows) per Decision 23.
embeddings = {}
for model in MODEL_NAMES:
    embeddings[(model, "f_clue_val")]     = np.load(EMBED_DIR / model / "f_clue_val.npy")
    embeddings[(model, "f_common_wndef")] = np.load(EMBED_DIR / model / "f_common_wndef.npy")

# Shape validation per CLAUDE.md — catches any file-swap mistakes.
EXPECTED = {
    "f_clue_val":     (47933, 1024),
    "f_common_wndef": (53930, 1024),
}
for (model, phrase), emb in embeddings.items():
    assert emb.shape == EXPECTED[phrase], (
        f"{model}/{phrase}: shape {emb.shape} != expected {EXPECTED[phrase]}"
    )

# f_clue_val index is identical across models per NB 04 checks; load once.
f_clue_index = pd.read_csv(
    EMBED_DIR / "g_stock" / "f_clue_val_index.csv",
    keep_default_na=False, na_values=[""],
)
clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        f_clue_index["clue_id"], f_clue_index["definition"], f_clue_index["row"],
    )
}
assert len(clue_key_to_row) == len(f_clue_index), (
    "Duplicate (clue_id, definition) keys in f_clue_val_index"
)
assert len(f_clue_index) == embeddings[("g_stock", "f_clue_val")].shape[0]
print(f"f_clue_val rows:  {len(f_clue_index):,} "
      f"(matches f_clue_val.npy for both models)")
print(f"Load time:        {time.time() - t0:.1f}s")

## §2 — Wordplay type landscape (validation set)

Before looking at ATE, characterize how frequent each wordplay type is within
the validation split. Two views: a per-type frequency table (row-level, so
double-def clues count once per expansion) and a co-occurrence heatmap
(clue_id-level, so a clue with two detections counts once).

The two views answer different questions. Row-level frequencies map directly
to the ATE subsample sizes in §4 — that is the denominator we care about when
asking "how confident can we be in this type's ATE." Clue-level co-occurrence
tells us whether the types are largely disjoint (so their ATEs are independent
stories) or overlapping (so a clue flagged as `anagram_consec` may also carry
other detections that contribute to its misdirection).

In [ ]:
# === Per-type row-level frequencies in the validation set
n_rows = len(joined)

# Count of rows with each type True. Row-level count aligns with the ATE
# subsample sizes used in §4 (a double-def clue contributes once per expansion).
freq_rows = [
    {
        "wordplay_type": col,
        "n_rows": int(joined[col].sum()),
        "pct_rows": float(joined[col].mean() * 100),
    }
    for col in WP_COLS
]
freq_df = pd.DataFrame(freq_rows).sort_values("n_rows", ascending=False).reset_index(drop=True)

any_detected_mask = joined[WP_COLS].any(axis=1)
none_detected_mask = ~any_detected_mask

print(f"Validation rows (total):        {n_rows:,}")
print(f"Rows with at least one type:    {int(any_detected_mask.sum()):,} "
      f"({any_detected_mask.mean() * 100:.1f}%)")
print(f"Rows with no type detected:     {int(none_detected_mask.sum()):,} "
      f"({none_detected_mask.mean() * 100:.1f}%)")
print()
with pd.option_context("display.float_format", "{:.2f}".format):
    print(freq_df.to_string(index=False))

### Co-occurrence heatmap (unique clue_ids)

For each pair of wordplay types (i, j), count the number of *unique clue_ids*
in the validation set where both type i and type j are True. The diagonal is
the count of unique clue_ids with that type True. Aggregating to clue_id
removes the double-counting introduced by multi-definition expansion — a
single clue is one mark no matter how many rows it produces.

In [ ]:
# === Co-occurrence heatmap (clue_id-level)
# Collapse to one row per clue_id: for each type, True if any row for that
# clue_id has the type True (they should all agree, since wordplay_metadata
# is per-clue_id and was broadcast across expansions).
per_clue = joined.groupby("clue_id", sort=False)[WP_COLS].any()
print(f"Unique validation clue_ids: {len(per_clue):,}")

# M is (K, N) where K = number of types, N = number of unique clue_ids.
# int32 is wide enough for all counts up to ~2B; int8 would silently overflow
# on types that occur more than 127 times (e.g. double_def has ~4,500 clues).
M = per_clue[WP_COLS].to_numpy().astype(np.int32).T
cooccur = M @ M.T  # shape (K, K); diagonal is per-type unique-clue_id count.

# Abbreviated labels for readability on the heatmap axes.
WP_ABBREV = {
    "anagram_single_word":  "anag_single",
    "anagram_consec_words": "anag_consec",
    "hidden_fwd":           "hid_fwd",
    "hidden_rev":           "hid_rev",
    "selection_alt":        "sel_alt",
    "selection_alt_rev":    "sel_alt_rev",
    "selection_firsts":     "sel_firsts",
    "selection_firsts_rev": "sel_firsts_rev",
    "selection_lasts":      "sel_lasts",
    "selection_lasts_rev":  "sel_lasts_rev",
    "double_def":           "double_def",
}
labels = [WP_ABBREV[c] for c in WP_COLS]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cooccur, annot=True, fmt="d", cmap="YlOrRd",
    xticklabels=labels, yticklabels=labels,
    cbar_kws={"label": "unique clue_ids"}, ax=ax,
)
ax.set_title("Wordplay Type Co-occurrence (Validation Set, unique clue_ids)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()

fig_path = FIGURE_DIR / "wp_cooccurrence_heatmap.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

# Keep the per-type unique clue_id counts for the results file later.
unique_clueid_per_type = pd.Series(
    np.diag(cooccur), index=WP_COLS, name="unique_clue_ids",
)

## §3 — Define analysis categories

Cryptic clues fall into two structural types. **Standard clues** have the
familiar definition + fodder + indicator structure, and the 10 algorithmically
detected letterplay types (anagram, hidden, selection, and their reversals)
describe what the fodder/indicator mechanism does. **Double-definition clues**
concatenate two or more definitions for the answer, each pointing to a
different sense — there is no fodder or indicator. `double_def` is therefore
a *structural* type, not a letterplay type, and its misdirection mechanism is
different from letterplay.

This two-level taxonomy drives the category definitions below:

1. **Structural (Level 1)** — `standard` vs `double_def`, spanning the full
   validation set. Drives Figure 3.
2. **Individual letterplay within standard clues (Level 2a)** — each of the
   10 letterplay columns intersected with `is_standard`. Drives Figure 1.
3. **Grouped letterplay within standard clues (Level 2b)** — unions across
   related letterplay columns plus `any_letterplay` / `no_letterplay`
   baselines, all intersected with `is_standard`. Drives Figure 2.

The intersection with `is_standard` is deliberate: a clue that is both
`double_def=True` and e.g. `anagram_consec=True` (there are a small number of
such co-occurrences) belongs to the double-def structural group, and mixing
it into letterplay breakdowns would confound the mechanisms. Those clues
appear in Figure 3 under `double_def` but are excluded from Figures 1 and 2.

`no_letterplay` — standard clues with no detected letterplay — serves as the
baseline for both letterplay figures. Comparing a letterplay type to
`no_letterplay` rather than to "all standard clues" isolates the effect of
that letterplay mechanism against the complement within the same structural
type.

Categories with fewer than 50 rows are flagged; ATE is still computed but
the small-N flag reminds the reader to treat those estimates as noisy.

In [ ]:
# === Define category masks and report sizes
SMALL_N_THRESHOLD = 50

# Letterplay columns = the 11 boolean columns minus double_def, which is a
# structural type. These 10 drive Figures 1 and 2.
LETTERPLAY_COLS = [c for c in WP_COLS if c != "double_def"]
assert len(LETTERPLAY_COLS) == 10

is_standard          = ~joined["double_def"]
any_letterplay_mask  = joined[LETTERPLAY_COLS].any(axis=1)

# Display names for the 10 letterplay types. Two are abbreviated
# (anagram_consec, anagram_single) to match the spec; the rest mirror the
# wordplay_metadata column name exactly.
LETTERPLAY_DISPLAY = {
    "anagram_consec_words":  "anagram_consec",
    "anagram_single_word":   "anagram_single",
    "hidden_fwd":            "hidden_fwd",
    "hidden_rev":            "hidden_rev",
    "selection_alt":         "selection_alt",
    "selection_alt_rev":     "selection_alt_rev",
    "selection_firsts":      "selection_firsts",
    "selection_firsts_rev":  "selection_firsts_rev",
    "selection_lasts":       "selection_lasts",
    "selection_lasts_rev":   "selection_lasts_rev",
}

categories = {}

# ---- Level 1: structural (Figure 3) ----
categories["standard"]   = is_standard
categories["double_def"] = joined["double_def"].copy()

# ---- Level 2a: individual letterplay within standard clues (Figure 1) ----
for col, name in LETTERPLAY_DISPLAY.items():
    categories[name] = is_standard & joined[col]

# ---- Level 2b: grouped letterplay within standard clues (Figure 2) ----
categories["any_anagram"] = is_standard & (
    joined["anagram_single_word"] | joined["anagram_consec_words"]
)
categories["any_hidden"] = is_standard & (
    joined["hidden_fwd"] | joined["hidden_rev"]
)
categories["any_reversal"] = is_standard & (
    joined["hidden_rev"]
    | joined["selection_alt_rev"]
    | joined["selection_firsts_rev"]
    | joined["selection_lasts_rev"]
)
categories["any_selection"] = is_standard & (
    joined["selection_alt"] | joined["selection_alt_rev"]
    | joined["selection_firsts"] | joined["selection_firsts_rev"]
    | joined["selection_lasts"] | joined["selection_lasts_rev"]
)
categories["any_letterplay"] = is_standard & any_letterplay_mask
categories["no_letterplay"]  = is_standard & ~any_letterplay_mask

# Level tag for §8's results-file grouping. Declared alongside the categories
# themselves so the grouping never drifts out of sync with the dict.
CATEGORY_LEVEL = {
    "standard":       "structural",
    "double_def":     "structural",
    **{name: "individual_letterplay" for name in LETTERPLAY_DISPLAY.values()},
    "any_anagram":    "grouped_letterplay",
    "any_hidden":     "grouped_letterplay",
    "any_reversal":   "grouped_letterplay",
    "any_selection":  "grouped_letterplay",
    "any_letterplay": "grouped_letterplay",
    "no_letterplay":  "grouped_letterplay",
}
assert set(CATEGORY_LEVEL) == set(categories), (
    "CATEGORY_LEVEL keys do not match categories keys"
)

cat_sizes = pd.DataFrame([
    {
        "category": name,
        "level":    CATEGORY_LEVEL[name],
        "n":        int(mask.sum()),
        "small_n":  bool(mask.sum() < SMALL_N_THRESHOLD),
    }
    for name, mask in categories.items()
])
print(f"Categories defined: {len(categories)} "
      f"(2 structural, 10 individual letterplay, 6 grouped letterplay)")
print(f"is_standard rows:   {int(is_standard.sum()):,} "
      f"({is_standard.mean() * 100:.1f}% of validation)")
print(f"double_def rows:    {int(joined['double_def'].sum()):,} "
      f"({joined['double_def'].mean() * 100:.1f}% of validation)")
print()
print(cat_sizes.to_string(index=False))
if cat_sizes["small_n"].any():
    small = cat_sizes[cat_sizes["small_n"]]["category"].tolist()
    print(f"\nSmall-N flag (< {SMALL_N_THRESHOLD}): {small}")

## §4 — ATE computation

The ATE is computed per row as

$$
\Delta = \cos(g(f\_\mathrm{clue}(\text{def})), g(f\_\mathrm{common}\_\mathrm{wndef}(\text{ans})))
\;-\;
\cos(g(f\_\mathrm{common}\_\mathrm{wndef}(\text{def})), g(f\_\mathrm{common}\_\mathrm{wndef}(\text{ans})))
$$

where the second term is the decontextualized baseline (T=0) and the first
is the clue-contextualized treatment (T=1). A negative Δ means embedding the
definition in its clue *weakens* its alignment with the answer — the
signature of surface-level misdirection. The ATE is the mean Δ within a
subset.

Lookups use the full-vocab wndef embeddings (Decision 23): every
`definition_wn` and `answer_wn` in the validation set is guaranteed to
resolve, and every (clue_id, definition) pair resolves via `f_clue_val_index`.
We assert this invariant rather than silently dropping rows.

For each category × model, a 1000-sample bootstrap on Δ (pinned seed)
produces a 95% CI on the mean.

In [ ]:
# === Resolve embedding-row indices for every validation row (done once)
# Using Decision 23 full-vocab wndef: every validation definition_wn / answer_wn
# lives in vocabulary_wndef, and every (clue_id, definition) lives in
# f_clue_val_index. We assert resolution rather than masking.
def_rows_wndef = np.array([
    wndef_word_to_row.get(w, -1) for w in joined["definition_wn"]
])
ans_rows_wndef = np.array([
    wndef_word_to_row.get(w, -1) for w in joined["answer_wn"]
])
clue_rows = np.array([
    clue_key_to_row.get((cid, defn), -1)
    for cid, defn in zip(joined["clue_id"], joined["definition"])
])

n_miss_def  = int((def_rows_wndef < 0).sum())
n_miss_ans  = int((ans_rows_wndef < 0).sum())
n_miss_clue = int((clue_rows       < 0).sum())
print(f"Unresolvable definition_wn: {n_miss_def}")
print(f"Unresolvable answer_wn:     {n_miss_ans}")
print(f"Unresolvable clue key:      {n_miss_clue}")
assert n_miss_def == 0 and n_miss_ans == 0 and n_miss_clue == 0, (
    "Full-vocab wndef should resolve every validation row per Decision 23"
)

def rowwise_cosine(A, B):
    """Per-row cosine similarity between two equal-shape (N, D) arrays."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

# Precompute T=0 and T=1 per row, per model, once — categories are just row
# masks over these vectors.
t0 = time.time()
per_row = {}
for model in MODEL_NAMES:
    clue_emb  = embeddings[(model, "f_clue_val")]
    wndef_emb = embeddings[(model, "f_common_wndef")]
    t0_vec = rowwise_cosine(wndef_emb[def_rows_wndef], wndef_emb[ans_rows_wndef])
    t1_vec = rowwise_cosine(clue_emb[clue_rows],       wndef_emb[ans_rows_wndef])
    per_row[model] = {"t0": t0_vec, "t1": t1_vec, "delta": t1_vec - t0_vec}
print(f"Precomputed per-row T=0 / T=1 / Δ for both models in "
      f"{time.time() - t0:.1f}s")
print(f"  delta vector length: {len(per_row['g_stock']['delta']):,} per model")

In [ ]:
# === ATE summary function + apply to every category × model
N_BOOTSTRAP = 1000

def compute_ate(mask, model, n_bootstrap=N_BOOTSTRAP, seed=RANDOM_STATE):
    """Summarize ATE on the rows selected by `mask` for a given model.

    Because T=0, T=1, and Δ are precomputed over all joined rows, this reduces
    to a slice-and-summarize: the per-row vectors are indexed by the mask and
    the resulting subset is bootstrapped for a CI on the mean.
    """
    idx = np.flatnonzero(mask.to_numpy())
    t0_sub = per_row[model]["t0"][idx]
    t1_sub = per_row[model]["t1"][idx]
    d_sub  = per_row[model]["delta"][idx]
    n = len(idx)
    if n == 0:
        return {
            "n_total": 0, "n_resolved": 0,
            "t0_mean": np.nan, "t1_mean": np.nan,
            "ate_mean": np.nan, "ate_median": np.nan,
            "ate_ci_lo": np.nan, "ate_ci_hi": np.nan,
            "pct_negative": np.nan,
        }
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_bootstrap)
    for b in range(n_bootstrap):
        sample_idx = rng.integers(0, n, size=n)
        boot_means[b] = d_sub[sample_idx].mean()
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
    return {
        "n_total":      n,
        "n_resolved":   n,  # Decision 23: full resolution guaranteed.
        "t0_mean":      float(t0_sub.mean()),
        "t1_mean":      float(t1_sub.mean()),
        "ate_mean":     float(d_sub.mean()),
        "ate_median":   float(np.median(d_sub)),
        "ate_ci_lo":    float(ci_lo),
        "ate_ci_hi":    float(ci_hi),
        "pct_negative": float((d_sub < 0).mean() * 100),
    }

t0 = time.time()
rows = []
for cat_name, mask in categories.items():
    for model in MODEL_NAMES:
        stats = compute_ate(mask, model)
        rows.append({
            "category": cat_name,
            "model":    model,
            **stats,
            "small_n":  stats["n_total"] < SMALL_N_THRESHOLD,
        })
ate_df = pd.DataFrame(rows)
print(f"Computed ATE for {len(categories)} categories × {len(MODEL_NAMES)} "
      f"models with {N_BOOTSTRAP}-sample bootstrap in "
      f"{time.time() - t0:.1f}s")

# Stable display sort: category name, then model.
display_df = ate_df.sort_values(["category", "model"]).reset_index(drop=True)
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 160):
    print(display_df.drop(columns=["n_resolved"]).to_string(index=False))

## §5 — ATE visualization

Three dot plots sharing the same visual conventions — both model markers on
the same y-tick, thin light-gray segment connecting them, 95% bootstrap CI
bars, dashed vertical line at Δ=0, N annotated on the right, small-N rows
rendered at reduced alpha. The plots differ only in which slice of the
category space they show.

- **Figure 1 — `wp_ate_individual_letterplay.png`:** `no_letterplay`
  baseline followed by the 10 individual letterplay types within standard
  clues, ordered by descending N so the well-powered estimates appear at the
  top.
- **Figure 2 — `wp_ate_grouped_letterplay.png`:** `no_letterplay` baseline
  followed by the grouped letterplay categories (`any_anagram`, `any_hidden`,
  `any_reversal`, `any_selection`, `any_letterplay`).
- **Figure 3 — `wp_ate_structural.png`:** the two structural categories —
  `standard` and `double_def`. This is the pair referenced from the §6
  deep-dive.

Splitting the plots lets each figure be read on its own without the scale or
visual density of the others getting in the way, and keeps each figure's
y-ordering meaningful (descending-N for Figure 1, fixed sequence for 2 & 3).

In [ ]:
# === ATE dot plot helper + three figures (individual, grouped, structural)
from matplotlib.lines import Line2D

MODEL_STYLE = {
    "g_stock": {"color": "tab:blue",   "marker": "o"},
    "g1":      {"color": "tab:orange", "marker": "^"},
}

def plot_ate_dotplot(cat_order, title, out_path):
    """One dot plot per figure; cat_order is top-to-bottom.

    All categories in cat_order must appear in ate_df (i.e., be defined in §3
    and have ATE computed in §4).
    """
    # Subset ate_df to just these categories, preserving the requested order.
    sub = ate_df[ate_df["category"].isin(cat_order)].copy()
    assert set(cat_order) == set(sub["category"]), (
        f"cat_order has entries not in ate_df: "
        f"{set(cat_order) - set(sub['category'])}"
    )

    # y grows upward in matplotlib; reversed() puts cat_order[0] at the top.
    y_positions = {name: i for i, name in enumerate(reversed(cat_order))}

    fig, ax = plt.subplots(figsize=(10, 0.45 * len(cat_order) + 1.5))

    # Thin light-gray pairing line at each category's y-tick, drawn first so
    # markers and CI bars land on top.
    sub_by_cm = sub.set_index(["category", "model"])
    for cat_name in cat_order:
        y = y_positions[cat_name]
        x_stock = sub_by_cm.loc[(cat_name, "g_stock"), "ate_mean"]
        x_g1    = sub_by_cm.loc[(cat_name, "g1"),      "ate_mean"]
        ax.plot(
            [x_stock, x_g1], [y, y],
            color="lightgray", linewidth=0.9, zorder=1,
        )

    for _, row in sub.iterrows():
        y = y_positions[row["category"]]
        style = MODEL_STYLE[row["model"]]
        alpha = 0.35 if row["small_n"] else 1.0
        linestyle = "--" if row["small_n"] else "-"
        ax.hlines(
            y=y, xmin=row["ate_ci_lo"], xmax=row["ate_ci_hi"],
            color=style["color"], alpha=alpha,
            linewidth=1.5, linestyle=linestyle, zorder=2,
        )
        ax.scatter(
            row["ate_mean"], y,
            color=style["color"], marker=style["marker"], s=55, alpha=alpha,
            edgecolors="black", linewidths=0.3, zorder=3,
        )

    # N annotations to the right of the CI extents.
    ns_by_cat = sub.drop_duplicates("category").set_index("category")["n_total"]
    xmax = sub["ate_ci_hi"].max()
    xmin = sub["ate_ci_lo"].min()
    x_pad = 0.05 * (xmax - xmin) if xmax > xmin else 0.01
    for cat_name in cat_order:
        y = y_positions[cat_name]
        ax.text(
            xmax + x_pad, y, f"N={int(ns_by_cat[cat_name]):,}",
            va="center", ha="left", fontsize=8, color="dimgray",
        )

    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_yticks(range(len(cat_order)))
    ax.set_yticklabels(list(reversed(cat_order)))
    ax.set_xlabel("ATE (mean Δ = T=1 − T=0)")
    ax.set_title(title)

    legend_handles = [
        Line2D([0], [0], color="tab:blue",   marker="o", linestyle="-",
               markeredgecolor="black", markersize=7, label="g_stock"),
        Line2D([0], [0], color="tab:orange", marker="^", linestyle="-",
               markeredgecolor="black", markersize=7, label="g1"),
        Line2D([0], [0], color="black", linestyle="--", label="Δ = 0"),
    ]
    ax.legend(handles=legend_handles, loc="lower right")

    ax.set_xlim(xmin - x_pad, xmax + 6 * x_pad)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    plt.show()
    print(f"Saved {out_path}")


# ---- Figure 1: individual letterplay, ordered by descending N ----
individual_names = list(LETTERPLAY_DISPLAY.values())
# Shared N per category (same for both models). Pull from ate_df once.
n_by_cat = ate_df.drop_duplicates("category").set_index("category")["n_total"]
individual_sorted = sorted(individual_names, key=lambda n: -int(n_by_cat[n]))
fig1_order = ["no_letterplay"] + individual_sorted
plot_ate_dotplot(
    fig1_order,
    "ATE by individual letterplay type — standard clues only "
    "(g_stock vs g1, 95% bootstrap CI)",
    FIGURE_DIR / "wp_ate_individual_letterplay.png",
)

# ---- Figure 2: grouped letterplay, fixed order ----
fig2_order = [
    "no_letterplay", "any_anagram", "any_hidden",
    "any_reversal", "any_selection", "any_letterplay",
]
plot_ate_dotplot(
    fig2_order,
    "ATE by grouped letterplay category — standard clues only "
    "(g_stock vs g1, 95% bootstrap CI)",
    FIGURE_DIR / "wp_ate_grouped_letterplay.png",
)

# ---- Figure 3: structural comparison ----
fig3_order = ["standard", "double_def"]
plot_ate_dotplot(
    fig3_order,
    "ATE by structural type — standard vs double-def "
    "(g_stock vs g1, 95% bootstrap CI)",
    FIGURE_DIR / "wp_ate_structural.png",
)

## §6 — Structural comparison: double-def vs standard

This section accompanies Figure 3 (the structural dot plot) with a deeper
look at the *distributional* difference between the two structural types,
not just their means. In a standard cryptic clue the tagged definition's
context is wordplay (fodder + indicator). In a double-def clue the tagged
definition's context is one or more *other* definitions of the same answer,
each typically pointing to a different sense; the concatenation may read as
a coherent phrase whose meaning has nothing to do with the answer
("Friendly drink" → CORDIAL). The misdirection, if any, comes from
sense-juxtaposition rather than wordplay. Whether this produces more, less,
or comparable ATE is an open empirical question — we compare the two
distributions directly.

In [ ]:
# === Double-def vs standard Δ distributions, per model
dd_mask  = categories["double_def"].to_numpy()
std_mask = categories["standard"].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for ax, model in zip(axes, MODEL_NAMES):
    deltas = per_row[model]["delta"]
    d_dd  = deltas[dd_mask]
    d_std = deltas[std_mask]

    # Common bins across both distributions so the shapes are directly
    # comparable at each Δ bucket.
    both = np.concatenate([d_dd, d_std])
    bins = np.linspace(both.min(), both.max(), 80)

    # density=True makes the two histograms comparable despite the ~10x
    # imbalance in sample size.
    ax.hist(d_std, bins=bins, density=True, alpha=0.55,
            label=f"standard (n={len(d_std):,}, mean={d_std.mean():.3f})",
            color="tab:blue")
    ax.hist(d_dd, bins=bins, density=True, alpha=0.55,
            label=f"double_def (n={len(d_dd):,}, mean={d_dd.mean():.3f})",
            color="tab:red")
    ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axvline(d_std.mean(), color="tab:blue", linewidth=1.2, linestyle=":")
    ax.axvline(d_dd.mean(),  color="tab:red",  linewidth=1.2, linestyle=":")
    ax.set_title(f"{model}: Δ distribution — standard vs double_def")
    ax.set_ylabel("density")
    ax.legend()
axes[-1].set_xlabel("Δ (T=1 − T=0)")
fig.tight_layout()

fig_path = FIGURE_DIR / "wp_double_def_comparison.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

# Collect the summary numbers referenced by the narrative cell + results file.
dd_summary = {}
for model in MODEL_NAMES:
    deltas = per_row[model]["delta"]
    dd_summary[model] = {
        "dd_mean":      float(deltas[dd_mask].mean()),
        "std_mean":     float(deltas[std_mask].mean()),
        "dd_median":    float(np.median(deltas[dd_mask])),
        "std_median":   float(np.median(deltas[std_mask])),
        "dd_pct_neg":   float((deltas[dd_mask] < 0).mean() * 100),
        "std_pct_neg":  float((deltas[std_mask] < 0).mean() * 100),
        "n_dd":         int(dd_mask.sum()),
        "n_std":        int(std_mask.sum()),
    }
for model, s in dd_summary.items():
    print(f"\n{model}:")
    print(f"  standard:   mean Δ = {s['std_mean']:+.4f}  "
          f"median Δ = {s['std_median']:+.4f}  "
          f"% Δ<0 = {s['std_pct_neg']:.1f}%  (n={s['n_std']:,})")
    print(f"  double_def: mean Δ = {s['dd_mean']:+.4f}  "
          f"median Δ = {s['dd_median']:+.4f}  "
          f"% Δ<0 = {s['dd_pct_neg']:.1f}%  (n={s['n_dd']:,})")

### Interpretation

Double-def clues show a clearly different Δ distribution from standard clues
under both models. Under g_stock, double-def mean ATE is −0.021 vs −0.068
for standard — roughly a third as much misdirection, with only 56% of deltas
negative (vs 73%). The distribution plot confirms this is a shift in
location, not just a tail effect: the entire double-def distribution sits to
the right of the standard distribution.

This difference is consistent with the structural distinction between the two
clue types. In a standard clue, the context surrounding the tagged definition
is wordplay — fodder and indicator material chosen to create a misleading
surface reading. In a double-def clue, the context is one or more other
definitions of the same answer, each pointing to a different sense. While the
concatenated surface may read as a coherent misleading phrase ("Friendly
drink" → CORDIAL), the individual definitions each independently relate to
the answer. The ATE is sensitive to this structural difference: clue context
that consists of other definitions produces less measured misdirection than
clue context that consists of wordplay.

Under g1, both distributions shift further negative (standard mean ATE:
−0.068 → −0.127; double_def mean ATE: −0.021 → −0.095), but the relative
ordering is preserved — double-def clues remain less negative. The gap
narrows somewhat (from 0.047 under g_stock to 0.032 under g1), consistent
with g1's uniform compression of decontextualized embeddings affecting both
structural types.

## §8 — Results file

Persist every number referenced above to a single markdown file so the
write-up and any downstream notebooks can cite stable values without
re-executing. The file is intentionally self-contained: full ATE table,
per-type frequency table, double-def comparison, and version stamps.

In [ ]:
# === Write results markdown file
results_path = OUTPUT_DIR / "wordplay_ate_breakdown-results.md"

def df_to_markdown_table(df, columns):
    """Minimal markdown-pipe table — avoids the optional tabulate dependency."""
    header = "| " + " | ".join(columns) + " |"
    sep    = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, r in df.iterrows():
        rows.append("| " + " | ".join(str(r[c]) for c in columns) + " |")
    return "\n".join([header, sep, *rows])

# Format ATE table cells for markdown, then carry `level` through so we can
# slice it per-section below.
ate_md = ate_df.copy()
for c in ["t0_mean", "t1_mean", "ate_mean", "ate_median", "ate_ci_lo", "ate_ci_hi"]:
    ate_md[c] = ate_md[c].map(lambda v: f"{v:+.4f}")
ate_md["pct_negative"] = ate_md["pct_negative"].map(lambda v: f"{v:.1f}%")
ate_md["small_n"]      = ate_md["small_n"].map({True: "yes", False: ""})
ate_md["n_total"]      = ate_md["n_total"].map(lambda v: f"{int(v):,}")
ate_md["level"]        = ate_md["category"].map(CATEGORY_LEVEL)

ate_md_cols = ["category", "model", "n_total", "t0_mean", "t1_mean",
               "ate_mean", "ate_median", "ate_ci_lo", "ate_ci_hi",
               "pct_negative", "small_n"]

# Per-level ordering mirrors the §5 figure ordering so the results file and
# the plots tell the same story in the same sequence.
structural_order   = ["standard", "double_def"]
individual_order   = ["no_letterplay"] + sorted(
    LETTERPLAY_DISPLAY.values(), key=lambda n: -int(n_by_cat[n]),
)
grouped_order      = ["no_letterplay", "any_anagram", "any_hidden",
                      "any_reversal", "any_selection", "any_letterplay"]

def ate_section(category_order):
    """Return rows in ate_md matching category_order (2 models × |order|),
    preserving the requested order and the g_stock-then-g1 model sort."""
    ordered = pd.concat(
        [ate_md[(ate_md["category"] == c) & (ate_md["model"] == m)]
         for c in category_order for m in MODEL_NAMES],
        ignore_index=True,
    )
    return ordered

freq_md = freq_df.copy()
freq_md["n_rows"]          = freq_md["n_rows"].map(lambda v: f"{int(v):,}")
freq_md["pct_rows"]        = freq_md["pct_rows"].map(lambda v: f"{v:.2f}%")
freq_md["unique_clue_ids"] = [f"{int(unique_clueid_per_type[t]):,}"
                              for t in freq_df["wordplay_type"]]
freq_md_cols = ["wordplay_type", "n_rows", "pct_rows", "unique_clue_ids"]

with open(results_path, "w") as f:
    f.write("# Wordplay ATE Breakdown — Results\n\n")
    f.write(f"Generated: {date.today().isoformat()}  \n")
    f.write(f"Notebook: `planning/exploration/wordplay_ate_breakdown.ipynb`  \n")
    f.write(f"Random seed: {RANDOM_STATE}  \n")
    f.write(f"Bootstrap samples: {N_BOOTSTRAP}  \n")
    f.write(f"Validation rows: {n_rows:,}  \n")
    f.write(f"Unique validation clue_ids: {len(per_clue):,}  \n")
    f.write(f"Standard clues (is_standard): {int(is_standard.sum()):,}  \n")
    f.write(f"Double-def clues: {int(joined['double_def'].sum()):,}  \n\n")

    f.write("## Per-type validation-set frequencies\n\n")
    f.write(f"- Rows with ≥1 type detected: {int(any_detected_mask.sum()):,} "
            f"({any_detected_mask.mean() * 100:.1f}%)\n")
    f.write(f"- Rows with no type detected: {int(none_detected_mask.sum()):,} "
            f"({none_detected_mask.mean() * 100:.1f}%)\n\n")
    f.write(df_to_markdown_table(freq_md, freq_md_cols))
    f.write("\n\n")

    f.write("## ATE by category × model\n\n")
    f.write("Δ = cos(g(f_clue(def)), g(wndef(ans))) − "
            "cos(g(wndef(def)), g(wndef(ans)))  \n")
    f.write("CI = 95% bootstrap CI on mean Δ. `small_n` marks categories "
            f"with fewer than {SMALL_N_THRESHOLD} rows. All letterplay rows "
            "are restricted to standard clues (is_standard).\n\n")

    f.write("### Structural\n\n")
    f.write(df_to_markdown_table(ate_section(structural_order), ate_md_cols))
    f.write("\n\n")

    f.write("### Individual letterplay (standard clues only)\n\n")
    f.write(df_to_markdown_table(ate_section(individual_order), ate_md_cols))
    f.write("\n\n")

    f.write("### Grouped letterplay (standard clues only)\n\n")
    f.write(df_to_markdown_table(ate_section(grouped_order), ate_md_cols))
    f.write("\n\n")

    f.write("## Structural comparison: double-def vs standard\n\n")
    for model, s in dd_summary.items():
        f.write(f"### {model}\n\n")
        f.write(f"- standard   (n={s['n_std']:,}): "
                f"mean Δ = {s['std_mean']:+.4f}, "
                f"median Δ = {s['std_median']:+.4f}, "
                f"% Δ<0 = {s['std_pct_neg']:.1f}%\n")
        f.write(f"- double_def (n={s['n_dd']:,}): "
                f"mean Δ = {s['dd_mean']:+.4f}, "
                f"median Δ = {s['dd_median']:+.4f}, "
                f"% Δ<0 = {s['dd_pct_neg']:.1f}%\n\n")

    f.write("## Version stamps\n\n")
    f.write(f"- pandas: {pd.__version__}\n")
    f.write(f"- numpy: {np.__version__}\n")
    f.write(f"- matplotlib: {matplotlib.__version__}\n")
    f.write(f"- seaborn: {sns.__version__}\n")

print(f"Wrote {results_path}")
print(f"  {results_path.stat().st_size:,} bytes")

## §7 — Summary

Exploratory ATE breakdown by algorithmically verifiable wordplay type on the
validation split, organized around a two-level taxonomy (structural, then
letterplay within standard clues).

- **Inputs:** `clues_val.csv` (47,933 validation rows), `wordplay_metadata.csv`
  (one row per source clue_id), full-vocab wndef embeddings (53,930 × 1024)
  and f_clue_val embeddings (47,933 × 1024) for both `g_stock` and `g1`.
- **Split:** standard clues vs double-def clues — see §3's printout for
  exact counts.
- **Categories analyzed:** 18 masks over the validation rows — 2 structural
  (`standard`, `double_def`), 10 individual letterplay types within standard
  clues, and 6 grouped letterplay categories within standard clues
  (`any_anagram`, `any_hidden`, `any_reversal`, `any_selection`,
  `any_letterplay`, `no_letterplay`). Each category's N is printed in §3;
  categories with N < 50 are marked in the ATE table and rendered at reduced
  alpha in the dot plots.
- **ATE methodology:** Per-row Δ = cos(g(f_clue(def)), g(wndef(ans))) −
  cos(g(wndef(def)), g(wndef(ans))). 1000-sample bootstrap CI on the mean,
  `random_state=42`. Decision-23 full-vocab wndef guarantees every
  validation row resolves — no rows dropped.
- **Key structural finding:** Double-def clues show substantially less
  misdirection than standard clues under g_stock (mean ATE = −0.021 vs
  −0.068; 56% vs 73% negative). This is consistent with the structural
  difference: double-def clue context consists of other definitions for the
  answer rather than wordplay. Under g1, both shift further negative but the
  gap persists (mean ATE −0.095 vs −0.127). See §6 for the distributional
  comparison.
- **Key letterplay findings:** Under g_stock, all well-powered letterplay
  types have mean ATEs in a narrow band (−0.057 to −0.071) around the
  `no_letterplay` mean ATE of −0.068, with no strong evidence that any
  individual letterplay mechanism produces meaningfully different
  misdirection. Under g1, every category's mean ATE shifts further negative,
  but the magnitude of that shift (g1 mean ATE minus g_stock mean ATE)
  varies: `no_letterplay` shifts by −0.063, while `anagram_consec` shifts
  by only −0.021. This differential may reflect compositional differences in
  the definition-answer word pairs across clue types rather than a genuine
  interaction between g1 and the wordplay mechanism — decomposing the ATE
  shift into its T=0 and T=1 components shows that T=0 (decontextualized,
  no access to the clue surface) accounts for most of the between-category
  variation. Controlling for word-pair properties would be needed to
  distinguish these explanations.
- **Figures produced** (300 dpi):
  - `outputs/figures/wp_cooccurrence_heatmap.png`
  - `outputs/figures/wp_ate_structural.png`
  - `outputs/figures/wp_ate_individual_letterplay.png`
  - `outputs/figures/wp_ate_grouped_letterplay.png`
  - `outputs/figures/wp_double_def_comparison.png`
- **Results file:** `outputs/wordplay_ate_breakdown-results.md` — ATE table
  grouped by level, per-type frequencies, and double-def summary numbers.
- **Runtime:** wall-clock times for the load step, per-row ATE
  precomputation, and the bootstrap loop are printed inline next to each
  step.